In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats
import time
import os
from tqdm import tqdm


path = str(os.getcwd())


In [3]:
MIN_TOTAL_COUNTS_FOR_CV = 100  # analysis restricted to higher-expression genes
N_FOLDS = 5
RANDOM_SEED = 0
MIN_COMPARTMENT_FRACTION = 0.05 # if any subregion has less than 5% of cell that brain is removed form analysis
META_COLS = ['cell_label', 'brain_section_label', 'supertype', 'subclass',
             'brain', 'EC', 'x_ccf', 'y_ccf', 'z_ccf']

In [6]:
merfish_full = pd.read_csv(f"{path}/input_data/merfish_final_obj_mesh_filter.csv", index_col=0)
gene_cols = [c for c in merfish_full.columns if c not in META_COLS]

composition = pd.crosstab(merfish_full['brain'], merfish_full['EC']) # count number of cell per region per brain
composition_frac = composition.div(composition.sum(axis=1), axis=0) # what fraction of all brain's cells a subregion is composed of

print(f"Subregion cell fraction per brain:\n{composition_frac}")


Subregion cell fraction per brain:
EC                  LEC       MEC     intEC
brain                                      
Zhuang-ABCA-1  0.282374  0.237410  0.480216
Zhuang-ABCA-2  0.274678  0.206009  0.519313
Zhuang-ABCA-3  0.185714  0.235714  0.578571
Zhuang-ABCA-4  0.000000  0.009709  0.990291


In [7]:
brains_to_exclude = [b for b in composition_frac.index if composition_frac.loc[b].min() < MIN_COMPARTMENT_FRACTION] 
merfish = merfish_full[~merfish_full['brain'].isin(brains_to_exclude)].copy()

print(f"Using {len(merfish)} cells from {sorted(merfish['brain'].unique())}")

Using 1209 cells from ['Zhuang-ABCA-1', 'Zhuang-ABCA-2', 'Zhuang-ABCA-3']


In [8]:
gene_total_counts = merfish[gene_cols].sum(axis=0) # total counts per gene
total_counts = merfish[gene_cols].sum(axis=1) # total counts per cell
log_offset = np.log(total_counts).values # cells total expresison is used as offset in log scale to focus on expression patterns not quantity differences

z_coord = merfish['z_ccf'].values # get mediolateral coordinate
z_coord_scaled = (z_coord - z_coord.mean()) / z_coord.std()
z_coord2_scaled = z_coord_scaled ** 2

# create brain and EC as dummies for linear model
brain_dummies = pd.get_dummies(merfish['brain'], drop_first=True).astype(float) 
ec_dummies = pd.get_dummies(merfish['EC'], drop_first=True).astype(float)


X_continuous = sm.add_constant(np.column_stack([z_coord_scaled, z_coord2_scaled, brain_dummies.values])) # parameters for continous model based on mediolateral position
X_discrete = sm.add_constant(np.column_stack([ec_dummies.values, brain_dummies.values])) # parameters for discrete model based on EC subregion

print(f"Continuous model params: {X_continuous.shape[1]}, Discrete model params: {X_discrete.shape[1]} "
      f"(should match for a fair comparison)")

Continuous model params: 5, Discrete model params: 5 (should match for a fair comparison)


### Comparison of continious and discrete GLMs for each gene

In [9]:
def poisson_loglik(y, X, params, offset):
    mu = np.exp(X @ params + offset)
    return np.sum(stats.poisson.logpmf(y, np.clip(mu, 1e-10, None)))

In [10]:
model_comparison_df_rows = []
for gene in tqdm(gene_cols):
    y = merfish[gene].values.astype(float)
    try:
        continuous_model = sm.GLM(y, X_continuous, family=sm.families.Poisson(), offset=log_offset).fit()
        discrete_model = sm.GLM(y, X_discrete, family=sm.families.Poisson(), offset=log_offset).fit()
        model_comparison_df_rows.append({'gene_id': gene, 'llf_continuous': continuous_model.llf, 'llf_discrete': discrete_model.llf,
                               'continuous_wins': continuous_model.llf > discrete_model.llf})
    except Exception:
        print("Cannot fit GLM for {gene}")
        pass
    

discrete_vs_continious_model = pd.DataFrame(model_comparison_df_rows)
discrete_vs_continious_model['total_count'] = discrete_vs_continious_model['gene_id'].map(gene_total_counts) 
continuous_model_win_fraction = discrete_vs_continious_model['continuous_wins'].mean() * 100
high_expression_continuous_model_win_fraction = discrete_vs_continious_model.loc[discrete_vs_continious_model['total_count'] >= MIN_TOTAL_COUNTS_FOR_CV, 'continuous_wins'].mean() * 100

print(f"\nIn-sample: continuous model preferred for {continuous_model_win_fraction:.0f}% of all {len(discrete_vs_continious_model)} genes, "
      f"{high_expression_continuous_model_win_fraction:.0f}% of {(discrete_vs_continious_model['total_count']>=MIN_TOTAL_COUNTS_FOR_CV).sum()} higher-expression genes")



  0%|          | 0/1122 [00:00<?, ?it/s]

100%|██████████| 1122/1122 [00:01<00:00, 580.82it/s]


In-sample: continuous model preferred for 65% of all 1122 genes, 75% of 528 higher-expression genes


### Cross-validation of discrete and continious using data splitting

In [11]:
high_expr_genes = discrete_vs_continious_model.loc[discrete_vs_continious_model['total_count'] >= MIN_TOTAL_COUNTS_FOR_CV, 'gene_id'].tolist()
print(f"Testing {len(high_expr_genes)} genes")

rng = np.random.default_rng(RANDOM_SEED)
n_cells = len(merfish)
fold_assignment = rng.permutation(n_cells) % N_FOLDS

continious_vs_discrete_cv_rows = []
for gene in tqdm(high_expr_genes):
    y = merfish[gene].values.astype(float)
    total_llf_continious, total_llf_discrete = 0.0, 0.0
    try:
        for fold in range(N_FOLDS):
            train_mask = fold_assignment != fold
            test_mask = fold_assignment == fold

            model_continious = sm.GLM(y[train_mask], X_continuous[train_mask], family=sm.families.Poisson(),
                             offset=log_offset[train_mask]).fit()
            model_discrete = sm.GLM(y[train_mask], X_discrete[train_mask], family=sm.families.Poisson(),
                             offset=log_offset[train_mask]).fit()

            total_llf_continious += poisson_loglik(y[test_mask], X_continuous[test_mask], model_continious.params, log_offset[test_mask])
            total_llf_discrete += poisson_loglik(y[test_mask], X_discrete[test_mask], model_discrete.params, log_offset[test_mask])

        continious_vs_discrete_cv_rows.append({'gene_id': gene, 'cv_llf_continuous': total_llf_continious, 'cv_llf_discrete': total_llf_discrete,
                         'continuous_wins_cv': total_llf_continious > total_llf_discrete})
    except Exception:
        pass
   

continious_vs_discrete_cv = pd.DataFrame(continious_vs_discrete_cv_rows)
continuous_wins = continious_vs_discrete_cv['continuous_wins_cv'].mean() * 100
print(f"\nCross-validated: continuous model preferred for {continuous_wins:.0f}% of {len(continious_vs_discrete_cv)} higher-expression genes")



Testing 528 genes


100%|██████████| 528/528 [00:04<00:00, 126.93it/s]


Cross-validated: continuous model preferred for 75% of 528 higher-expression genes


### Continious and discrete model comparisons for gene categories from main analysis

In [14]:

try:
    classification = pd.read_csv(f'{path}/output_data/merfish_final_gene_categories.csv')
    continious_vs_discrete_cv = continious_vs_discrete_cv.merge(classification[['gene_id', 'category']], on='gene_id', how='left')
    monotonic = continious_vs_discrete_cv[continious_vs_discrete_cv['category'].isin(['medial-enriched', 'lateral-enriched'])]
    nonmonotonic = continious_vs_discrete_cv[continious_vs_discrete_cv['category'].isin(['middle-peak', 'middle-trough'])]
    if len(monotonic):
        print(f"Monotonic genes (n={len(monotonic)}): continuous preferred for "
              f"{monotonic['continuous_wins_cv'].mean()*100:.0f}%")
    if len(nonmonotonic):
        print(f"Non-monotonic genes (n={len(nonmonotonic)}): continuous preferred for "
              f"{nonmonotonic['continuous_wins_cv'].mean()*100:.0f}%")
except FileNotFoundError:
    print("merfish_final_gene_categories.csv not found -- run the main classification pipeline first "
          "to enable the monotonic/non-monotonic split.")

# save the model comparisons as files
discrete_vs_continious_model.to_csv('continuous_vs_discrete.csv', index=False)
continious_vs_discrete_cv.to_csv('continuous_vs_discrete_cross-validation.csv', index=False)

# Aggregate held-out log-likelihood advantage WITHOUT the DV covariate
# (sum across genes of continuous - discrete CV log-likelihood), for
# comparison against the DV-covariate version below.
aggregate_advantage_no_dv = (continious_vs_discrete_cv['cv_llf_continuous'] - continious_vs_discrete_cv['cv_llf_discrete']).sum()
print(f"\nAggregate held-out log-likelihood advantage (continuous - discrete), no DV covariate: "
      f"{aggregate_advantage_no_dv:.0f}")



Monotonic genes (n=271): continuous preferred for 83%
Non-monotonic genes (n=148): continuous preferred for 82%

Aggregate held-out log-likelihood advantage (continuous - discrete), no DV covariate: 8814


### Dorsoventral covariate sensitivity check

In [15]:
y_coord = merfish['y_ccf'].values # dorsoventral (dv) coordinate
y_coord_scaled = (y_coord - y_coord.mean()) / y_coord.std()

X_continuous_dv = sm.add_constant(np.column_stack([z_coord_scaled, z_coord2_scaled, y_coord_scaled, brain_dummies.values]))
X_discrete_dv = sm.add_constant(np.column_stack([ec_dummies.values, y_coord_scaled, brain_dummies.values]))
print(f"Continuous+DV model params: {X_continuous_dv.shape[1]}, "
      f"Discrete+DV model params: {X_discrete_dv.shape[1]} (should still match)")

continious_vs_discrete_cv_dv_rows = []
for gene in tqdm(high_expr_genes):
    y = merfish[gene].values.astype(float)
    total_llf_continious, total_llf_discrete = 0.0, 0.0
    try:
        for fold in range(N_FOLDS):
            train_mask = fold_assignment != fold
            test_mask = fold_assignment == fold

            model_continious = sm.GLM(y[train_mask], X_continuous_dv[train_mask], family=sm.families.Poisson(),
                             offset=log_offset[train_mask]).fit()
            model_discrete = sm.GLM(y[train_mask], X_discrete_dv[train_mask], family=sm.families.Poisson(),
                             offset=log_offset[train_mask]).fit()

            total_llf_continious += poisson_loglik(y[test_mask], X_continuous_dv[test_mask], model_continious.params, log_offset[test_mask])
            total_llf_discrete += poisson_loglik(y[test_mask], X_discrete_dv[test_mask], model_discrete.params, log_offset[test_mask])

        continious_vs_discrete_cv_dv_rows.append({'gene_id': gene, 'cv_llf_continuous_dv': total_llf_continious,
                            'cv_llf_discrete_dv': total_llf_discrete,
                            'continuous_wins_cv_dv': total_llf_continious > total_llf_discrete})
    except Exception:
        pass
    

continious_vs_discrete_cv_dv = pd.DataFrame(continious_vs_discrete_cv_dv_rows)
continuous_wins_dv = continious_vs_discrete_cv_dv['continuous_wins_cv_dv'].mean() * 100
aggregate_advantage_dv = (continious_vs_discrete_cv_dv['cv_llf_continuous_dv'] - continious_vs_discrete_cv_dv['cv_llf_discrete_dv']).sum()

print(f"\nWith y_ccf covariate: continuous model preferred for {continuous_wins_dv:.1f}% "
      f"of {len(continious_vs_discrete_cv_dv)} higher-expression genes")
print(f"Aggregate held-out log-likelihood advantage (continuous - discrete), WITH DV covariate: "
      f"{aggregate_advantage_dv:.0f}")
print(f"\nComparison:")
print(f"  Without DV covariate: {continuous_wins:.1f}% of genes, aggregate advantage {aggregate_advantage_no_dv:.0f}")
print(f"  With DV covariate:    {continuous_wins_dv:.1f}% of genes, aggregate advantage {aggregate_advantage_dv:.0f}")
print(f"\n(Methods text reported: 71.0% with DV vs. 73.1% without; "
      f"aggregate advantage 6,579 with DV vs. 8,398 without)")

continious_vs_discrete_cv_dv.to_csv('continuous_vs_discrete_dorsoventral_covariate.csv', index=False)
print("\nSaved: continuous_vs_discrete_dorsoventral_covariate.csv")


Continuous+DV model params: 6, Discrete+DV model params: 6 (should still match)


100%|██████████| 528/528 [00:04<00:00, 113.95it/s]


With y_ccf covariate: continuous model preferred for 73.5% of 528 higher-expression genes
Aggregate held-out log-likelihood advantage (continuous - discrete), WITH DV covariate: 6981

Comparison:
  Without DV covariate: 74.6% of genes, aggregate advantage 8814
  With DV covariate:    73.5% of genes, aggregate advantage 6981

(Methods text reported: 71.0% with DV vs. 73.1% without; aggregate advantage 6,579 with DV vs. 8,398 without)

Saved: continuous_vs_discrete_dorsoventral_covariate.csv
